# Train, save, reload, and evaluate

This Colab notebook calls the same package functions as the command-line scripts.
Model weights, configuration, logs, and predictions share one experiment directory.
The previous executed experiment is preserved in `archive/jev_20261018_pilot.ipynb`.

Select a CUDA GPU runtime. Set the checkout and dataset paths below. Training is explicit;
the optional Jev API comparison is disabled by default.

In [ ]:
import importlib
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
REPO_ROOT = Path("/content/drive/MyDrive/jev_project/jev-like-decision-tuning")
assert (REPO_ROOT / "pyproject.toml").is_file(), "Set REPO_ROOT to the full repository checkout"
source_root = REPO_ROOT / "src"
for filename in ("__init__.py", "data.py", "prompts.py", "modeling.py", "training.py", "evaluation.py", "experiment.py"):
    assert (source_root / "jev_decision_tuning" / filename).is_file(), f"Missing source file: {filename}"

install_target = f"{REPO_ROOT}[train]"
%pip install -q -e "$install_target"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
importlib.invalidate_caches()

## Configure one experiment

The three JSON files differ only in `target_group`: `attention`, `mlp`, or `attention_mlp`.
For this first run, keep the combined configuration. Data and option seeds stay fixed;
`train_seed` controls adapter initialization and training randomness.

In [ ]:
from datetime import datetime, timezone
from jev_decision_tuning.config import load_config
from jev_decision_tuning.experiment import train_experiment, evaluate_experiment

DATA_PATH = Path("/content/drive/MyDrive/jev_project/data/tweet_eval_sentiment")
OUTPUT_ROOT = Path("/content/drive/MyDrive/jev_project/runs")
config = load_config(REPO_ROOT / "configs/attention_mlp.json", data_path=str(DATA_PATH))
run_name = f"{config.target_group}_seed{config.train_seed}_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
RUN_DIR = OUTPUT_ROOT / run_name
print(config.to_dict())
print("Run directory:", RUN_DIR)

## Train and save

This cell trains a fresh adapter on 9,000 selected examples and saves it under `RUN_DIR`.
An existing run directory is never overwritten. This is a new training run, not optimizer-state resume.

In [ ]:
RUN_DIR = train_experiment(config, RUN_DIR)
print("Completed run:", RUN_DIR)
print("Keep this exact path to evaluate after restarting the runtime.")

## Reload independently and evaluate

The following calls load the base model and saved adapter from disk; they do not use an
in-memory training model. After a runtime restart, run the setup cell and the imports below,
then set `RUN_DIR` to the exact completed path printed above. Do not rerun training.

Each evaluation name is saved once. To deliberately rerun an evaluation, supply a new `name`.
`split="train"` means the full training split, not only the selected 9,000 training examples.

In [ ]:
from pathlib import Path
from jev_decision_tuning.experiment import evaluate_experiment
# After restarting, uncomment and replace this with your saved run path:
# RUN_DIR = Path("/content/drive/MyDrive/jev_project/runs/your_completed_run")

base_report = evaluate_experiment(RUN_DIR, backend="base", split="validation")
lora_report = evaluate_experiment(RUN_DIR, backend="lora", split="validation")

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {"model": label, **{key: report[key] for key in ("accuracy", "macro_f1", "macro_recall")}}
    for label, report in (("Base Qwen", base_report), ("LoRA", lora_report))
]).set_index("model")
display(comparison)
print("LoRA confusion matrix (rows=true, columns=predicted):")
display(pd.DataFrame(lora_report["confusion_matrix"], index=lora_report["label_names"], columns=lora_report["label_names"]))

## Optional test evaluation

Enable only after the experiment configuration is fixed. Evaluate both the base comparator
and the adapter on the same test rows. These calls can also run in a new runtime.

In [ ]:
RUN_TEST_EVALUATION = False
if RUN_TEST_EVALUATION:
    test_base_report = evaluate_experiment(RUN_DIR, backend="base", split="test")
    test_lora_report = evaluate_experiment(RUN_DIR, backend="lora", split="test")
    print("Base test macro recall:", test_base_report["macro_recall"])
    print("LoRA test macro recall:", test_lora_report["macro_recall"])

## Optional Jev comparison

Install the `jev` extra (for example, use `[train,jev]` in the setup cell) and add a Colab
Secret named `jev` before enabling this cell. This makes external API requests.
The new cache has a request/dataset manifest; old pilot checkpoints are not reused automatically.

In [ ]:
RUN_JEV = False
if RUN_JEV:
    import json
    from google.colab import userdata
    from jev_decision_tuning.data import load_sentiment_dataset
    from jev_decision_tuning.evaluation import evaluate_jev, save_evaluation

    saved_config = json.loads((RUN_DIR / "config.json").read_text())
    validation = load_sentiment_dataset(saved_config["data_path"])["validation"]
    records = evaluate_jev(
        validation,
        api_key=userdata.get("jev"),
        checkpoint_path=RUN_DIR / "jev_cache/validation.jsonl",
    )
    jev_report = save_evaluation(
        records, RUN_DIR / "evaluation", name="validation_jev",
        metadata={"backend": "jev", "model": "jev-1.13.0", "split": "validation"},
    )
    print(jev_report)